In [27]:
import pickle
import os
from pathlib import Path

import sympy as sp
from IPython.display import display, Math
sp.init_printing()
DIR = Path(os.getcwd())

# Single pendulum derivation

<center>
<img src='./media/cartpole-cart-single.svg' style='width: 30%; height: auto;' alt='Single pendulum'>
</center>

In [28]:
import sympy as sp
import sympy.physics.mechanics as mech

m, M, l_1, g, m_1, J_1 = sp.symbols('m M l_1 g m_1 J_1')

t = mech.dynamicsymbols._t

q1, q2 = mech.dynamicsymbols('x theta')
q1d, q2d = mech.dynamicsymbols('x theta', 1)

u, w = mech.dynamicsymbols('u w')   # control action and disturbance

# Inertial frame
N = mech.ReferenceFrame('N')

# rod_frame.y points along pendulum, rotate inertial frame q2 about N.z
rod_frame = N.orientnew("R", "Axis", (q2, N.z))

# Origin
O = mech.Point('O')
O.set_vel(N, 0)

# Cart point (COM for M)
P_M = O.locatenew('P_M', q1*N.x)
P_M.set_vel(N, q1d*N.x)

# Rod center of mass, halfway along rod
P_R = P_M.locatenew("P_R", (l_1/2)*rod_frame.y)
P_R.set_vel(N, P_R.pos_from(O).dt(N))

# Pendulum end point (COM for m)
P_m = P_M.locatenew(
    'P_m',
    -l_1*sp.sin(q2)*N.x + l_1*sp.cos(q2)*N.y
)
P_m.set_vel(N, P_m.pos_from(O).dt(N))


rod_inertia = mech.inertia(rod_frame, 0, 0, J_1)
rod = mech.RigidBody("rod", P_R, rod_frame, m_1, (rod_inertia, P_R))

# Particles
cart = mech.Particle('cart', P_M, M)
pole_end = mech.Particle('bob', P_m, m)
bodies = [cart, rod, pole_end]

# Potential energy
cart.potential_energy = 0
pole_end.potential_energy = m*g*l_1*sp.cos(q2)
rod.potential_energy = m_1*g*(l_1/2)*sp.cos(q2)

# Lagrangian
L = mech.Lagrangian(N, cart, rod, pole_end)

# Non-conservative force on cart
forcelist = [
    (P_M, (u + w)*N.x)  # control u and a disturbance w acts on the COM of M
]

LM = mech.LagrangesMethod(
    L,
    [q1, q2],
    forcelist=forcelist,
    frame=N,
    bodies=bodies
)

eom = LM.form_lagranges_equations()
z = LM.q.col_join(LM._qdots)    # sympy stores state as q, q'

In [29]:
def display_sym_expr(expr: sp.Basic, size: str='Large', simplify: bool=False):
    # Symbols for printed / state-space form. Remove the explicit time dependence x(t) -> x, and
    # replace d/dt notation with dots
    x_s = sp.Symbol('x')
    theta_s = sp.Symbol(r'\theta')

    xd_s = sp.Symbol(r'\dot{x}')
    thetad_s = sp.Symbol(r'\dot\theta')

    xdd_s = sp.Symbol(r'\ddot{x}')
    thetadd_s = sp.Symbol(r'\ddot\theta')

    # Important: substitute higher derivatives first
    pretty_subs = [
        (sp.diff(q1, t, 2), xdd_s),
        (sp.diff(q2, t, 2), thetadd_s),
        (sp.diff(q1, t), xd_s),
        (sp.diff(q2, t), thetad_s),
        (q1, x_s),
        (q2, theta_s),
    ]
    expr_pretty = expr.subs(pretty_subs)
    if simplify:
        expr_pretty = sp.simplify(expr_pretty)
    display(Math(rf'\{size} ' + sp.latex(expr_pretty)))

In [30]:
print('Implicit EOM:')
display_sym_expr(eom)

print('Mass matrix:')
display_sym_expr(LM.mass_matrix)

print('Forcing:')
display_sym_expr(LM.forcing)

print('Explicit EOMs')
rhs_exp = sp.simplify(LM.rhs())
display_sym_expr(sp.Equality(sp.diff(z, t), rhs_exp))

Implicit EOM:


<IPython.core.display.Math object>

Mass matrix:


<IPython.core.display.Math object>

Forcing:


<IPython.core.display.Math object>

Explicit EOMs


<IPython.core.display.Math object>

### Linearized dynamics

##### We are not interested in the full linearization expression about some linearization point, only the derivatives with respect to our state $z$ and control action $u$.

In [31]:
# Reorder variables as z = x, x', theta, theta' to agree with function calls in dynamics.py
P = sp.Matrix([
    [1, 0, 0, 0],
    [0, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
])
z = P * z
f = P * rhs_exp

df_dz = f.jacobian(z)
df_du = f.jacobian([u])

display(Math(r'$\Large z$'))
display_sym_expr(z)
display(Math(r'$\Large f, \quad \dot z = f(z, \dot z)$'))
display_sym_expr(f)
display(Math(r'$\Large \frac {df} {dz}$'))
display_sym_expr(df_dz)
display(Math(r'$\Large \frac {df} {du}$'))
display_sym_expr(df_du)


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

##### We are also interested in the energy in the system

In [32]:
T = mech.kinetic_energy(N, *LM.bodies)
V = mech.potential_energy(*LM.bodies)


print('Kinetic energy')
display_sym_expr(T)
print('Potential energy')
display_sym_expr(V)

Kinetic energy


<IPython.core.display.Math object>

Potential energy


<IPython.core.display.Math object>

### Export dynamics

##### Convert the dynamic symbols (time dependent functions) and replace them by a symbol. We want to export the formulas with non-time dependent state symbols, as we want to evaluate the expression at some value of z, u, etc. and not take time into consideration

In [33]:
# this should cover all of the dynamic symbols
x_s, x_t_s, x_tt_s = sp.symbols('x x_t x_tt')
th_s, th_t_s, th_tt_s = sp.symbols('theta theta_t theta_tt')
u_s, w_s = sp.symbols('u w')
dyn_to_static = {
    q1.diff(t, 2): x_tt_s,
    q2.diff(t, 2): th_tt_s,
    q1.diff(t): x_t_s,
    q2.diff(t): th_t_s,
    q1: x_s,
    q2: th_s,
    u: u_s,
    w: w_s,
}

def get_symbols(expr: sp.Basic):
    return expr.free_symbols, mech.find_dynamicsymbols(expr)


all_symbols = set()

exported_expr = {'f': f, 'df_dz': df_dz, 'df_du': df_du, 'Ek': T, 'Ep': V, 'state_symbols': z,
    'state_dot_symbols': sp.diff(z, t), 'u_symbol': u, 'w_symbol': w}

for expr_name in exported_expr.keys():
    exported_expr[expr_name] = exported_expr[expr_name].subs(dyn_to_static)
    all_symbols.update(*get_symbols(exported_expr[expr_name]))


exported_expr['func_symbols'] = all_symbols

assert t not in all_symbols, 'Failed to remove all time dependency from sympy expressions'    

with open(DIR / 'cart_pole' / 'dynamics_single.pkl', 'wb') as outfile:
    pickle.dump(exported_expr, outfile)


for (expr_name, expr) in exported_expr.items():
    print(f'Free and dynamic symbols in {expr_name}')
    try:
        display(get_symbols(expr))
    except:
        display(expr)



Free and dynamic symbols in f


Free and dynamic symbols in df_dz


Free and dynamic symbols in df_du


Free and dynamic symbols in Ek


Free and dynamic symbols in Ep


Free and dynamic symbols in state_symbols


Free and dynamic symbols in state_dot_symbols


Free and dynamic symbols in u_symbol


Free and dynamic symbols in w_symbol


Free and dynamic symbols in func_symbols
